# Gemma Grounded Health: training run

Thin runner (D-10): repo at `59e04afe17ce`, backend `unsloth`, 300 steps (pilot). All logic lives in the tested `ggh` package; any failing step stops the run.

In [ ]:
import os, subprocess, shutil, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4 (D-9), even on a T4 x2 machine
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd):
    """Run a shell command; a non-zero exit stops the notebook."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

sh("git clone -q https://github.com/biplob819/gemma-grounded-health /kaggle/working/ggh")
os.chdir("/kaggle/working/ggh")
sh("git checkout -q 59e04afe17ce7a37f25271492fc39f60f09521cf")
sh("pip install -q -r notebooks/requirements-kaggle-unsloth.txt")
sh("pip install -q --no-deps -e .")
sh("pip install -q --no-deps 'evalkit-harness @ git+https://github.com/biplob819/evalkit@06591f8de9b178078c971940a83019a6530fd119'")

In [ ]:
sh("pip list 2>/dev/null | grep -i -E '^(torch|transformers|peft|accelerate|bitsandbytes|unsloth|unsloth.zoo|evalkit-harness) ' || true")
sh("ggh doctor --kaggle --work /kaggle/working")

In [ ]:
# rebuild N1.2 data and replay N1.3 targets; every committed hash is checked
sh("ggh data")
sh("git diff --exit-code docs/measurements/data_manifest.json")
sh("ggh targets --replay")

In [ ]:
sh("ggh train --config configs/recipe.yaml --out /kaggle/working/run --tool unsloth --max-steps 300")

In [ ]:
report = json.load(open("/kaggle/working/run/report.json"))
shutil.copy("/kaggle/working/run/report.json", "/kaggle/working/report.json")
short = {k: v for k, v in report.items() if k not in ('losses', 'eval_losses')}
print(json.dumps(short, indent=2))